In [1]:
!pip install pandas scikit-learn joblib

In [3]:
# ============================================================
# CLASIFICADOR AUTOMÁTICO DE REQUISITOS DE SOFTWARE
# Metodología: Knowledge Discovery in Text - KDT
# Modelo: TF-IDF + Regresión Logística
# Aprendizaje: Supervisado
# ============================================================
import pandas as pd
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re


In [4]:
# 1. Cargar dataset e iniciar limpieza

#df = pd.read_csv("requisitos.csv")
df = pd.read_excel("dataset.xlsx")

print("Primeros registros:")
print(df.head())
print("\nDistribución de clases:")
print(df["tipo_requisito"].value_counts())


Primeros registros:
                                    sentence_espanol tipo_requisito
0  El sistema creará un único registro de pacient...             RF
1  El sistema asociará (almacenará y vinculará) i...             RF
2  El sistema deberá brindar la capacidad de alma...             RF
3  El sistema proporcionará un campo que identifi...             RF
4  El sistema deberá brindar la capacidad de fusi...             RF

Distribución de clases:
tipo_requisito
RF     3754
RNF     970
Name: count, dtype: int64


In [5]:


# 2. Eliminar valores vacíos y duplicados

# Cantidad inicial de registros
total_inicial = len(df)

# Contar valores vacíos en las columnas importantes
vacios = df[["sentence_espanol", "tipo_requisito"]].isna().any(axis=1).sum()

# Eliminar valores vacíos
df = df.dropna(subset=["sentence_espanol", "tipo_requisito"])

# Contar duplicados en sentence_espanol
duplicados = df.duplicated(subset=["sentence_espanol"]).sum()

# Eliminar duplicados
df = df.drop_duplicates(subset=["sentence_espanol"])

# Cantidad final
total_final = len(df)

# Mostrar resultados
print("=== LIMPIEZA DEL DATASET ===")
print(f"Registros iniciales:        {total_inicial}")
print(f"Registros con valores vacíos: {vacios}")
print(f"Registros duplicados:        {duplicados}")
print(f"Registros finales:           {total_final}")

=== LIMPIEZA DEL DATASET ===
Registros iniciales:        4724
Registros con valores vacíos: 0
Registros duplicados:        4
Registros finales:           4720


In [6]:
# 3. Limpiar texto

def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(r"[^a-záéíóúüñ0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

# Crear columna con el texto limpio
df["texto_limpio"] = df["sentence_espanol"].apply(limpiar_texto)

# Contar cuántas filas cambiaron
filas_limpiadas = (df["sentence_espanol"].astype(str) != df["texto_limpio"]).sum()

# Mostrar resultados
print("=== LIMPIEZA DE TEXTO ===")
print(f"Total de filas:              {len(df)}")
print(f"Filas que fueron limpiadas:  {filas_limpiadas}")
print(f"Filas sin cambios:            {len(df) - filas_limpiadas}")

# Mostrar el DataFrame
df

=== LIMPIEZA DE TEXTO ===
Total de filas:              4720
Filas que fueron limpiadas:  4720
Filas sin cambios:            0


,sentence_espanol,tipo_requisito,texto_limpio
0,El sistema creará un único registro de pacient...,RF,el sistema creará un único registro de pacient...
1,El sistema asociará (almacenará y vinculará) i...,RF,el sistema asociará almacenará y vinculará inf...
2,El sistema deberá brindar la capacidad de alma...,RF,el sistema deberá brindar la capacidad de alma...
3,El sistema proporcionará un campo que identifi...,RF,el sistema proporcionará un campo que identifi...
4,El sistema deberá brindar la capacidad de fusi...,RF,el sistema deberá brindar la capacidad de fusi...
...,...,...,...
4719,El sistema debe permitir modificar la política...,RF,el sistema debe permitir modificar la política...
4720,El sistema debe permitir comentar la inversión...,RF,el sistema debe permitir comentar la inversión...
4721,El sistema debe permitir programar la direcció...,RF,el sistema debe permitir programar la direcció...
4722,El sistema debe proteger el video de análisis ...,RNF,el sistema debe proteger el video de análisis ...


In [7]:
# 4. Variables
x = df["texto_limpio"]
y = df["tipo_requisito"]
x

,texto_limpio
0,el sistema creará un único registro de pacient...
1,el sistema asociará almacenará y vinculará inf...
2,el sistema deberá brindar la capacidad de alma...
3,el sistema proporcionará un campo que identifi...
4,el sistema deberá brindar la capacidad de fusi...
...,...
4719,el sistema debe permitir modificar la política...
4720,el sistema debe permitir comentar la inversión...
4721,el sistema debe permitir programar la direcció...
4722,el sistema debe proteger el video de análisis ...


In [8]:
# 5. División 80/20
#Indica que el 20 % de los datos será utilizado para evaluación y el 80 % para entrenamiento.
# random_state=42 hace que la selección aleatoria de los datos sea siempre la misma cada vez que ejecuto el programa.

X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.20, random_state=42, stratify=y )


In [9]:
# 6. Pipeline
modelo = Pipeline([
  ("tfidf", TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    max_features=10000,
    strip_accents="unicode"
  )),
  ("clasificador", LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
  ))
])

In [10]:
# 7. Entrenamiento

print("Entrenando modelo...")
modelo.fit(X_train, y_train)
print("Modelo entrenado correctamente.")

Entrenando modelo...
Modelo entrenado correctamente.


In [11]:
# 8. Predicción sobre el conjunto de prueba
y_pred = modelo.predict(X_test)

In [12]:
# 9. Evaluación
accuracy = accuracy_score(y_test, y_pred)
print("\n==============================")
print("RESULTADOS")
print("==============================")
print(f"Accuracy: {accuracy:.2%}")
print("\nReporte de clasificación:\n")
print(classification_report(y_test, y_pred, zero_division=0))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))


RESULTADOS
Accuracy: 96.29%

Reporte de clasificación:

              precision    recall  f1-score   support

          RF       0.99      0.97      0.98       750
         RNF       0.88      0.95      0.91       194

    accuracy                           0.96       944
   macro avg       0.93      0.96      0.94       944
weighted avg       0.96      0.96      0.96       944


Matriz de confusión:
[[725  25]
 [ 10 184]]


In [13]:
# 10. Guardar modelo
joblib.dump(modelo, "clasificador_requisitos.pkl")
print("Modelo guardado en clasificador_requisitos.pkl")

Modelo guardado en clasificador_requisitos.pkl


In [14]:
# 11. Probar nuevos requisitos
"""
nuevos_requisitos = [
"El sistema deberá permitir registrar nuevos estudiantes",
"Solo el administrador podrá eliminar cuentas de usuario",
"La plataforma deberá responder en menos de dos segundos",
"La aplicación deberá ser compatible con lectores de pantalla",
"La interfaz debe ser sencilla e intuitiva"
]
"""

nuevos_requisitos = [
"El sistema debe permitir al atleta consultar su historial deportivo (evaluaciones, competencias y progresos)",
"El sistema debe permitir al administrador modificar los roles de los usuarios",
"El sistema debe permitir al usuario consultar su propio historial deportivo",
"El sistema debe mantener una interfaz intuitiva basada en estándares UX/UI",
"El sistema debe cargar el historial deportivo en un tiempo máximo de 2 segundos"
]
print("\n==============================")
print("NUEVAS PREDICCIONES")
print("==============================\n")
for requisito in nuevos_requisitos:
  requisito_limpio = limpiar_texto(requisito)
  prediccion = modelo.predict([requisito_limpio])[0]
  probabilidades = modelo.predict_proba([requisito_limpio])[0]
  mejor_probabilidad = max(probabilidades)
  print("Requisito:", requisito)
  print("Clasificación:", prediccion)
  print(f"Confianza: {mejor_probabilidad:.2%}")
  print("-" * 60)


NUEVAS PREDICCIONES

Requisito: El sistema debe permitir al atleta consultar su historial deportivo (evaluaciones, competencias y progresos)
Clasificación: RF
Confianza: 87.25%
------------------------------------------------------------
Requisito: El sistema debe permitir al administrador modificar los roles de los usuarios
Clasificación: RF
Confianza: 79.05%
------------------------------------------------------------
Requisito: El sistema debe permitir al usuario consultar su propio historial deportivo
Clasificación: RF
Confianza: 94.99%
------------------------------------------------------------
Requisito: El sistema debe mantener una interfaz intuitiva basada en estándares UX/UI
Clasificación: RNF
Confianza: 65.28%
------------------------------------------------------------
Requisito: El sistema debe cargar el historial deportivo en un tiempo máximo de 2 segundos
Clasificación: RNF
Confianza: 86.84%
------------------------------------------------------------
